# Unidade 1 - Bloco prático da Aula 02: métricas sob desbalanceamento

Compara um classificador trivial, uma regressão logística e uma floresta aleatória num problema binário sintético com 5% de positivos, usando seis métricas lado a lado (acurácia, precisão, revocação, F1, MCC, AUC). Semente fixa (`seed=42`).

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)

# Problema binario desbalanceado: 5% de positivos
X, y = make_classification(
    n_samples=5000, n_features=10, n_informative=5,
    weights=[0.95, 0.05], flip_y=0.01, random_state=42
)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

modelos = {
    "Trivial (classe majoritaria)": DummyClassifier(strategy="most_frequent"),
    "Regressao logistica": LogisticRegression(max_iter=1000),
    "Floresta aleatoria": RandomForestClassifier(random_state=42),
}

print(f"{'Modelo':<30}{'Acur.':>7}{'Prec.':>7}{'Rec.':>7}"
      f"{'F1':>7}{'MCC':>7}{'AUC':>7}")
for nome, modelo in modelos.items():
    modelo.fit(X_tr, y_tr)
    pred = modelo.predict(X_te)
    if hasattr(modelo, "predict_proba"):
        score = modelo.predict_proba(X_te)[:, 1]
        auc = roc_auc_score(y_te, score)
    else:
        auc = 0.5
    print(f"{nome:<30}"
          f"{accuracy_score(y_te, pred):>7.3f}"
          f"{precision_score(y_te, pred, zero_division=0):>7.3f}"
          f"{recall_score(y_te, pred):>7.3f}"
          f"{f1_score(y_te, pred):>7.3f}"
          f"{matthews_corrcoef(y_te, pred):>7.3f}"
          f"{auc:>7.3f}")